## Sessão de IA tradicional
### Como prever uma partida sem ser chute?

Nesta sessão vamos construir um modelo preditivo para a Copa do Mundo 2026. A ideia não é começar pelo algoritmo, mas por uma pergunta simples:

> Se você tivesse que prever Brasil x Argentina, que informações olharia antes de dar um palpite?

Essas informações viram **features**. O resultado real dos jogos antigos vira **label**. O modelo aprende com o histórico e depois gera uma **predição** para uma partida nova.

Ao longo do notebook você vai passar por este fluxo:

1. carregar o histórico de jogos;
2. limpar e organizar os dados;
3. criar variáveis que existem antes da partida;
4. treinar modelos para resultado e gols;
5. avaliar se o modelo generaliza;
6. encapsular a predição em uma função reutilizável.

Conceitos técnicos que vamos usar sem perder a intuição:

| Termo | Explicação prática |
| --- | --- |
| Dataset | A tabela histórica de jogos |
| Feature | Informação conhecida antes da partida |
| Label | Resultado real usado como gabarito |
| Treino | Parte em que o modelo estuda o histórico |
| Teste | Parte escondida para medir se o modelo aprendeu de verdade |
| Predição | Probabilidade ou valor estimado para uma nova partida |
| Inferência | Uso do modelo já treinado em uma nova pergunta |

---

### Acessando dados

Os dados foram carregados na sessão anterior na camada bronze. Pense nessa camada como o dado bruto: ele existe, mas ainda precisa ser organizado antes de virar entrada para um modelo.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

DATA_PATH_CANDIDATES = [
    Path("worldcup_matches.csv"),
    Path("DeepDive Workshop/worldcup_matches.csv"),
    Path("oracle/DeepDive Workshop/worldcup_matches.csv"),
]

DATA_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Arquivo worldcup_matches.csv não encontrado. "
        "Execute o notebook a partir da pasta DeepDive Workshop ou da raiz do repo."
    )

try:
    spark
except NameError:
    spark = None

if spark is not None:
    try:
        bronze_df = spark.table("deepdivecatalog_bronze.admin.bronze_wc_matches")
        display(bronze_df.limit(5))
    except Exception as exc:
        print(f"Tabela Spark não disponível ({exc}). Usando CSV local: {DATA_PATH}")
        bronze_pd = pd.read_csv(DATA_PATH, encoding="latin1")
        display(bronze_pd.head())
else:
    print(f"Ambiente Spark não detectado. Usando CSV local: {DATA_PATH}")
    bronze_pd = pd.read_csv(DATA_PATH, encoding="latin1")
    display(bronze_pd.head())

### Carregar e limpar dados

Antes de falar em modelo, precisamos garantir que a tabela esteja consistente. Nesta etapa vamos padronizar nomes de colunas, tratar campos vazios e converter datas.

Em linguagem simples: estamos saindo de uma tabela bruta para uma versão mais confiável, que será salva na camada prata.


In [ ]:
import re
import pandas as pd


def normalize_column_name(column_name: str) -> str:
    return re.sub(r"_+", "_", re.sub(r"[^0-9a-zA-Z]+", "_", column_name.strip().lower())).strip("_")


if "bronze_pd" in globals():
    silver_df = bronze_pd.copy()
    silver_df.columns = [normalize_column_name(col) for col in silver_df.columns]

    silver_df = silver_df.replace("", pd.NA)
    silver_df["match_date"] = pd.to_datetime(silver_df["match_date"], format="%m/%d/%Y", errors="coerce")

    numeric_columns = [
        "home_team_score",
        "away_team_score",
        "home_team_win",
        "away_team_win",
        "draw",
    ]
    for column in numeric_columns:
        silver_df[column] = pd.to_numeric(silver_df[column], errors="coerce")

    silver_df = silver_df.drop_duplicates()
    silver_df = silver_df.dropna(subset=[
        "match_date",
        "home_team_name",
        "away_team_name",
        "home_team_score",
        "away_team_score",
        "home_team_win",
        "away_team_win",
        "draw",
    ])

    for column in ["home_team_score", "away_team_score", "home_team_win", "away_team_win", "draw"]:
        silver_df[column] = silver_df[column].astype(int)

    display(silver_df.head())
else:
    from pyspark.sql.functions import col, to_date, when

    normalized_columns = [normalize_column_name(c) for c in bronze_df.columns]
    silver_df = (
        bronze_df
        .toDF(*normalized_columns)
        .select([
            when(col(c) == "", None).otherwise(col(c)).alias(c)
            for c in normalized_columns
        ])
        .withColumn("match_date", to_date(col("match_date"), "M/d/yyyy"))
        .dropDuplicates()
        .dropna(subset=[
            "match_date",
            "home_team_name",
            "away_team_name",
            "home_team_score",
            "away_team_score",
            "home_team_win",
            "away_team_win",
            "draw",
        ])
    )

    display(silver_df.limit(5))

In [ ]:
if hasattr(silver_df, "write"):
    (
        silver_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("deepdivecatalog_prata.silver_wc_matches")
    )
else:
    print("Execução local: silver_df está em pandas; gravação Delta foi pulada.")

### Preparação para o modelo - Feature engineering

**Feature engineering** é o processo de transformar dados brutos em variáveis que o modelo consegue usar.

Na prática da Copa, uma feature é uma informação que estaria disponível **antes** do jogo acontecer: força relativa, média de gols, desempenho histórico, taxa de vitória etc.

Regra importante: nenhuma feature deve usar informação que só seria conhecida depois da partida. Isso evita vazamento de dados e deixa a avaliação honesta.

#### Convertendo para pandas

A partir daqui vamos trabalhar em pandas para facilitar a criação das variáveis e o treinamento com `scikit-learn`.


In [ ]:
if isinstance(silver_df, pd.DataFrame):
    df_pd = silver_df.sort_values("match_date").copy()
else:
    df_pd = silver_df.orderBy("match_date").toPandas()

display(df_pd.head())

#### Criando a variável de ELO

O **ELO** é uma forma de representar a força relativa de um time a partir do histórico de resultados.

A ideia é simples:

- todos os times começam com uma pontuação inicial;
- quando um time vence, sua pontuação sobe;
- quando perde, sua pontuação cai;
- vencer um adversário forte muda mais o ELO do que vencer um adversário fraco.

Para a apresentação, pense no ELO como uma variável que resume a pergunta: **quem chega mais forte para essa partida?**

Tecnicamente, ele vira uma feature numérica que o modelo pode combinar com outras variáveis.


In [ ]:
def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def update_elo(rating, expected, actual, k=30):
    return rating + k * (actual - expected)


def compute_elo(df, k=30):

    df = df.sort_values("match_date").copy()

    ratings = {}
    elo_home = []
    elo_away = []

    for _, row in df.iterrows():

        home = row["home_team_name"]
        away = row["away_team_name"]

        ratings.setdefault(home, 1500)
        ratings.setdefault(away, 1500)

        rating_home = ratings[home]
        rating_away = ratings[away]

        elo_home.append(rating_home)
        elo_away.append(rating_away)

        expected_home = expected_score(rating_home, rating_away)
        expected_away = expected_score(rating_away, rating_home)

        if row["home_team_win"] == 1:
            actual_home, actual_away = 1, 0
        elif row["away_team_win"] == 1:
            actual_home, actual_away = 0, 1
        else:
            actual_home = actual_away = 0.5

        ratings[home] = update_elo(rating_home, expected_home, actual_home, k)
        ratings[away] = update_elo(rating_away, expected_away, actual_away, k)

    df["elo_home"] = elo_home
    df["elo_away"] = elo_away
    df["elo_diff"] = df["elo_home"] - df["elo_away"]

    return df, ratings


df, ratings = compute_elo(df_pd)

#### Criando métricas históricas para usar como features

Além do ELO, vamos criar estatísticas agregadas por time, como média de gols, média de gols sofridos e taxa de vitória.

Essas variáveis ajudam o modelo a enxergar padrões que uma pessoa também olharia antes de prever uma partida.


In [ ]:
import pandas as pd
def create_team_features(df):

    goals_for = pd.concat([
        df[["home_team_name", "home_team_score"]].rename(
            columns={"home_team_name": "Team", "home_team_score": "Goals_For"}
        ),
        df[["away_team_name", "away_team_score"]].rename(
            columns={"away_team_name": "Team", "away_team_score": "Goals_For"}
        )
    ])

    goals_against = pd.concat([
        df[["home_team_name", "away_team_score"]].rename(
            columns={"home_team_name": "Team", "away_team_score": "Goals_Against"}
        ),
        df[["away_team_name", "home_team_score"]].rename(
            columns={"away_team_name": "Team", "home_team_score": "Goals_Against"}
        )
    ])

    wins = pd.concat([
        df[["home_team_name", "home_team_win"]].rename(
            columns={"home_team_name": "Team", "home_team_win": "Win"}
        ),
        df[["away_team_name", "away_team_win"]].rename(
            columns={"away_team_name": "Team", "away_team_win": "Win"}
        )
    ])

    team_stats = goals_for.groupby("Team").mean()
    team_stats["avg_goals_against"] = goals_against.groupby("Team").mean()
    team_stats["win_rate"] = wins.groupby("Team").mean()

    return team_stats.fillna(0)


team_stats = create_team_features(df)

In [ ]:
def add_match_features(df, team_stats):

    df = df.copy()

    df = df.merge(
        team_stats,
        left_on="home_team_name",
        right_index=True,
        how="left"
    ).rename(columns={
        "Goals_For": "teamA_avg_goals",
        "avg_goals_against": "teamA_avg_conceded",
        "win_rate": "teamA_win_rate"
    })

    df = df.merge(
        team_stats,
        left_on="away_team_name",
        right_index=True,
        how="left",
        suffixes=("", "_B")
    ).rename(columns={
        "Goals_For": "teamB_avg_goals",
        "avg_goals_against": "teamB_avg_conceded",
        "win_rate": "teamB_win_rate"
    })

    df["goals_diff"] = df["teamA_avg_goals"] - df["teamB_avg_goals"]
    df["defense_diff"] = df["teamA_avg_conceded"] - df["teamB_avg_conceded"]
    df["win_rate_diff"] = df["teamA_win_rate"] - df["teamB_win_rate"]

    return df


df = add_match_features(df, team_stats)

#### Preparação do dataset

Agora vamos montar a tabela final de treino.

Aqui aparece a separação mais importante da IA tradicional supervisionada:

- `X`: as features, ou seja, as informações disponíveis antes do jogo;
- `y`: o label, ou seja, o resultado real usado como gabarito.

Neste notebook, o resultado da partida será tratado como uma tarefa de **classificação**: vitória do mandante, empate ou vitória do visitante.


In [ ]:
def prepare_training_data(df: pd.DataFrame):

    features = [
        "elo_diff",
        "goals_diff",
        "defense_diff",
        "win_rate_diff"
    ]

    # Classes: empate -> 0, vitoria do mandante -> 1, vitoria do visitante -> 2
    df["match_result"] = (
        df["home_team_win"] * 1 +
        df["away_team_win"] * 2
    ).astype(int)

    training_df = df.dropna(subset=features + ["match_result", "home_team_score", "away_team_score"])

    X = training_df[features]
    y_result = training_df["match_result"]
    y_home_goals = training_df["home_team_score"].astype(float)
    y_away_goals = training_df["away_team_score"].astype(float)

    return X, y_result, y_home_goals, y_away_goals


X, y_result, y_home, y_away = prepare_training_data(df)
print(X.shape, y_result.shape)

#### Treinando o modelo

Treinar um modelo é como separar estudo e prova.

O modelo estuda uma parte dos jogos antigos, chamada de **treino**, e depois é avaliado em uma parte escondida, chamada de **teste**. Isso mede se ele aprendeu um padrão que generaliza ou se apenas decorou o histórico.

Normalmente vemos divisões como `70/30`, mas aqui vamos usar `80/20` para dar mais dados ao treino, já que o conjunto é pequeno.

Também vamos comparar mais de um algoritmo. O objetivo não é decorar o funcionamento interno de cada um, mas perceber que diferentes modelos podem ser avaliados com a mesma métrica.


In [ ]:
import sklearn

print(f"scikit-learn disponível: {sklearn.__version__}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_result,
    test_size=0.2,
    random_state=42,
    stratify=y_result
)

models = {
    "logreg": LogisticRegression(max_iter=2000),
    "rf": RandomForestClassifier(n_estimators=300, random_state=42)
}

for model in models.values():
    model.fit(X_train, y_train)

#### Avaliação do modelo com os 20% dos dados escondidos

Depois de treinar, precisamos medir.

A pergunta aqui é: **nos jogos que ficaram escondidos, quantas vezes o modelo acertou?**

Essa primeira avaliação usa `accuracy`, uma métrica simples que mede a proporção de previsões corretas. Ela é uma boa porta de entrada, mas não conta a história inteira quando as classes estão desequilibradas.


In [ ]:
for name, model in models.items():
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"{name}: {acc:.4f}")

#### Modelo preditivo de gols

Além de prever a classe do resultado, também vamos estimar quantidade de gols.

Aqui mudamos o tipo de problema:

- prever vitória, empate ou derrota é **classificação**;
- prever número de gols é **regressão**.

Essa distinção é um ponto técnico importante da IA tradicional: o tipo de resposta esperado define o tipo de modelo e a métrica de avaliação.


In [ ]:
home_goals_model = RandomForestRegressor(n_estimators=300, random_state=42)
home_goals_model.fit(X, y_home)

away_goals_model = RandomForestRegressor(n_estimators=300, random_state=42)
away_goals_model.fit(X, y_away)

## Testando com dados da Copa do Mundo de 2022

Agora vamos fazer uma avaliação mais intuitiva: usar a Copa de 2022 como referência.

A pergunta é: **se o modelo fosse usado para prever jogos desse período, como ele teria se saído?**

Isso ajuda a conectar o conceito de generalização com um cenário que a audiência reconhece.


In [ ]:
df["year"] = pd.to_datetime(df["match_date"]).dt.year
df_2022 = df[df["year"] == 2022].dropna(subset=[
    "elo_diff",
    "goals_diff",
    "defense_diff",
    "win_rate_diff",
    "match_result",
])

if df_2022.empty:
    raise ValueError("Não há jogos de 2022 disponíveis para avaliação.")

X_2022 = df_2022[["elo_diff", "goals_diff", "defense_diff", "win_rate_diff"]]
y_2022 = df_2022["match_result"].astype(int)

model = models["rf"]

preds_2022 = model.predict(X_2022)

print("Accuracy Copa 2022:", accuracy_score(y_2022, preds_2022))

### Matriz de confusão

A matriz de confusão é o placar dos acertos e erros por classe.

Ela mostra quando o modelo acertou e quando confundiu uma classe com outra. Para uma audiência iniciante, leia assim:

- acertei quando previ uma classe e ela aconteceu;
- errei quando previ uma classe e outra aconteceu;
- os erros mostram onde o modelo se confunde mais.

Esse é o caminho para métricas como precision, recall e F1-score.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

labels = [0, 1, 2]
display_labels = ["Draw", "Home Win", "Away Win"]

cm = confusion_matrix(y_2022, preds_2022, labels=labels)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=display_labels
)

disp.plot(cmap="Blues", values_format="d")
plt.title("Matriz de Confusão - Copa 2022")
plt.show()

#### Correlação das variáveis

A correlação ajuda a observar como as variáveis se relacionam entre si e com o resultado.

Ela não prova causalidade, mas é útil para investigar se uma feature tem sinal informativo para o modelo.

Na prática: queremos entender se variáveis como diferença de ELO, diferença de gols e taxa de vitória parecem conversar com a variável alvo.


In [ ]:
import matplotlib

print(f"matplotlib disponível: {matplotlib.__version__}")

In [ ]:
import seaborn as sns

print(f"seaborn disponível: {sns.__version__}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

features = [
    "elo_diff",
    "goals_diff",
    "defense_diff",
    "win_rate_diff",
    "match_result"   # target
]

corr = df[features].corr()

plt.figure(figsize=(8,6))
sns.heatmap(
    corr,
    annot=True,       # mostra valores
    fmt=".2f",        # 2 casas decimais
    cmap="coolwarm",  
    square=True
)

plt.title("Matriz de Correlação")
plt.show()

#### Encapsulando o modelo para predição

Até aqui treinamos e avaliamos o modelo. Agora vamos transformar a inferência em uma função reutilizável.

Esse é o ponto que conecta IA tradicional com agentes: o agente não precisa saber treinar o modelo. Ele só precisa chamar uma função/tool com entradas claras e receber uma saída clara.

Entrada esperada: dois times.

Saída esperada: probabilidade de vitória, empate, derrota e placar estimado.


In [ ]:
ratings_global = ratings
team_stats_global = team_stats
model_global = models["rf"]
home_goals_model_global = home_goals_model
away_goals_model_global = away_goals_model

In [ ]:
import pandas as pd

def predict_match_simple(home_team, away_team):

    rating_home = ratings_global.get(home_team, 1500)
    rating_away = ratings_global.get(away_team, 1500)

    elo_diff = rating_home - rating_away

    teamA = team_stats_global.loc[home_team].to_dict() if home_team in team_stats_global.index else {
        "Goals_For": 1.2, "avg_goals_against": 1.2, "win_rate": 0.5
    }

    teamB = team_stats_global.loc[away_team].to_dict() if away_team in team_stats_global.index else {
        "Goals_For": 1.2, "avg_goals_against": 1.2, "win_rate": 0.5
    }

    features = pd.DataFrame([{
        "elo_diff": rating_home - rating_away,
        "goals_diff": teamA["Goals_For"] - teamB["Goals_For"],
        "defense_diff": teamA["avg_goals_against"] - teamB["avg_goals_against"],
        "win_rate_diff": teamA["win_rate"] - teamB["win_rate"]
    }])

    probs = model_global.predict_proba(features)[0]

    home_goals = home_goals_model_global.predict(features)[0]
    away_goals = away_goals_model_global.predict(features)[0]

    class_mapping = dict(zip(model_global.classes_, probs))

    return {
        "home_win": float(class_mapping.get(1, 0)),
        "draw": float(class_mapping.get(0, 0)),
        "away_win": float(class_mapping.get(2, 0)),
        "home_goals": float(home_goals),
        "away_goals": float(away_goals)
    }

#### Predizer a partida entre Brasil e Argentina

Agora que a função de predição está pronta, vamos testar com dois times conhecidos.

Importante: leia o resultado como **probabilidade calculada**, não como certeza. O modelo usa apenas as variáveis que criamos e os dados disponíveis no histórico.


In [ ]:
result = predict_match_simple("Brazil", "Argentina")

print(f"Brazil win: {result['home_win']:.2%}")
print(f"Draw: {result['draw']:.2%}")
print(f"Argentina win: {result['away_win']:.2%}")
print(f"Score: {result['home_goals']:.1f} x {result['away_goals']:.1f}")

#### Desafios

1. Faça a predição de diferentes times que você conhece e veja se o resultado parece razoável.
2. Adicione uma nova feature e avalie se o modelo melhora ou piora.
3. Teste outro algoritmo do `scikit-learn` e compare a métrica.
4. Explique em linguagem simples o que mudou entre classificação e regressão neste notebook.
5. Pense em como essa função poderia virar uma tool chamada por um agente.
